In [2]:
import streamlit as st
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import sys
# import seaborn as sns

sys.path.append("/Users/toby/Dev/lionel-app/")
from lionel_app.connector import DBManager

from lionel_app.plot_team import create_plot, create_value_plot

In [3]:

dbm = DBManager("/Users/toby/Dev/lionel/data/fpl.db")
# Session = sessionmaker(bind=dbm.engine)

In [96]:
df_players = pd.DataFrame(dbm.query("SELECT * FROM player_inference").all())
df_players['color'] = df_players['position'].apply(lambda x: {'GK': '#abb8f1', 'DEF': '#818cb6', 'MID': '#58617b', 'FWD': 'black'}[x])
df_players['mean_minutes'] = df_players['mean_minutes'].round(0)
df_players

,player_name,position,team_name,goals_scored,assists,mean_minutes,color
0,A.Becker,GK,Liverpool,0.001,0.001,76.0,#abb8f1
1,A.Davies,GK,Sheffield Utd,0.015,0.026,0.0,#abb8f1
2,A.Doucoure,MID,Everton,0.182,0.098,73.0,#58617b
3,A.Fatawu,MID,Leicester,0.092,0.161,7.0,#58617b
4,A.Murphy,DEF,Newcastle,0.042,0.053,0.0,#818cb6
...,...,...,...,...,...,...,...
1137,Zych,GK,Aston Villa,0.018,0.026,0.0,#abb8f1
1138,Álvarez,MID,West Ham,0.044,0.046,67.0,#58617b
1139,Ângelo,MID,Chelsea,0.108,0.119,0.0,#58617b
1140,Ødegaard,MID,Arsenal,0.095,0.126,88.0,#58617b


In [112]:
min_minutes=45
df_plot = df_players[df_players.mean_minutes > min_minutes]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_plot[df_plot.position=="GK"]['assists'],
        y=df_plot[df_plot.position=="GK"]['goals_scored'],
        mode='markers',
        marker=dict(color='#abb8f1'),
        customdata=df_plot[df_plot.position=="GK"][['player_name', 'position', 'team_name', 'mean_minutes']],
        hovertemplate="<b>%{customdata[0]}</b><br>Position: %{customdata[1]}<br>Team: %{customdata[2]}<br>Avg Minutes: %{customdata[3]}<br>Goals: %{x}<br>Assists: %{y}",
        name="Goalkeepers",
        showlegend=True
    )
)
fig.add_trace(
    go.Scatter(
        x=df_plot[df_plot.position=="DEF"]['assists'],
        y=df_plot[df_plot.position=="DEF"]['goals_scored'],
        mode='markers',
        marker=dict(color='#818cb6'),
        customdata=df_plot[df_plot.position=="DEF"][['player_name', 'position', 'team_name', 'mean_minutes']],
        hovertemplate="<b>%{customdata[0]}</b><br>Position: %{customdata[1]}<br>Team: %{customdata[2]}<br>Avg Minutes: %{customdata[3]}<br>Goals: %{x}<br>Assists: %{y}",
        name="Defenders",
        showlegend=True
    )
)
fig.add_trace(
    go.Scatter(
        x=df_plot[df_plot.position=="MID"]['assists'],
        y=df_plot[df_plot.position=="MID"]['goals_scored'],
        mode='markers',
        marker=dict(color='#58617b'),
        customdata=df_plot[df_plot.position=="MID"][['player_name', 'position', 'team_name', 'mean_minutes']],
        hovertemplate="<b>%{customdata[0]}</b><br>Position: %{customdata[1]}<br>Team: %{customdata[2]}<br>Avg Minutes: %{customdata[3]}<br>Goals: %{x}<br>Assists: %{y}",
        name="Midfielders",
        showlegend=True
    )
)

fig.add_trace(
    go.Scatter(
        x=df_plot[df_plot.position=="FWD"]['assists'],
        y=df_plot[df_plot.position=="FWD"]['goals_scored'],
        mode='markers',
        marker=dict(color='black'),
        customdata=df_plot[df_plot.position=="FWD"][['player_name', 'position', 'team_name', 'mean_minutes']],
        hovertemplate="<b>%{customdata[0]}</b><br>Position: %{customdata[1]}<br>Team: %{customdata[2]}<br>Avg Minutes: %{customdata[3]}<br>Goals: %{x}<br>Assists: %{y}",
        name="Forwards",
        showlegend=True
    )
)
fig.update_layout(
    # add horizontal legend
    legend_orientation="h",
    legend=dict(x=0.1, y=-0.3),
    xaxis_title="Assist Probability",
    yaxis_title="Goal Probability",
)
fig.show()


# Team plot

In [120]:
df_team_inf = pd.DataFrame(dbm.query("SELECT * FROM team_inference").all())
df_team_inf[['attack', 'defence']] = df_team_inf[['attack', 'defence']]/100
df_team_inf

,team_name,attack,defence
0,Arsenal,0.399339,-0.410217
1,Aston Villa,0.247323,0.016129
2,Bournemouth,-0.064805,0.062899
3,Brentford,-0.019801,0.037693
4,Brighton,-0.035360,-0.030524
5,Burnley,-0.222755,0.222625
6,Chelsea,0.269979,0.025315
7,Crystal Palace,-0.049721,-0.024690
8,Everton,-0.254724,-0.044003
9,Fulham,-0.070399,-0.016856


In [123]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_team_inf['attack'],
        y=df_team_inf['defence'],
        mode='markers',
        marker=dict(color='#58617b'),
        customdata=df_team_inf[['team_name', 'attack', 'defence']],
        hovertemplate=(
            "<b>%{customdata[0]}</b>" +             
            "<br>Attack: %{customdata[1]:.1%}" + 
            "<br>Defence: %{customdata[2]:.1%}"
        ),
        name=""
    )
)
fig.update_layout(
    xaxis_title="Attack Strength",
    yaxis_title="Defence Strength",
)

In [4]:
df = pd.DataFrame(dbm.query("SELECT * from next_games").fetchall())

In [10]:
b(df['home_team'] + " vs " + df['away_team']).to_list()

['Aston Villa vs Wolves',
 'Brighton vs Nottingham',
 'Crystal Palace vs Manchester Utd',
 'Fulham vs Newcastle',
 'Leicester vs Everton',
 'Liverpool vs Bournemouth',
 'Manchester City vs Arsenal',
 'Southampton vs Ipswich',
 'Tottenham vs Brentford',
 'West Ham vs Chelsea']

# Scoreline plot

In [37]:
df_ = pd.DataFrame(dbm.query("SELECT * from scorelines").fetchall())
df_ = df_[(df_['home'] == 'Manchester City') & (df_['away'] == 'Arsenal')]

In [39]:
df_.to_csv("/Users/toby/Dev/lionel-app/data/city_arsenal.csv", index=False)
df_

,match,home_goals,away_goals,chain,draw,home,away
24000,426,0,0,0,0,Manchester City,Arsenal
24001,426,0,0,0,1,Manchester City,Arsenal
24002,426,3,0,0,2,Manchester City,Arsenal
24003,426,3,2,0,3,Manchester City,Arsenal
24004,426,2,3,0,4,Manchester City,Arsenal
...,...,...,...,...,...,...,...
27995,426,2,0,3,995,Manchester City,Arsenal
27996,426,1,1,3,996,Manchester City,Arsenal
27997,426,0,0,3,997,Manchester City,Arsenal
27998,426,0,3,3,998,Manchester City,Arsenal


In [19]:
from lionel_app.plot_team import build_scoreline_plot

In [20]:
f = build_scoreline_plot(df_, 'Manchester City', "Arsenal")

In [24]:
f.write_json("/Users/toby/Dev/lionel-app/data/city_arsenal.json")

In [31]:
import plotly
import re
import html

In [33]:
fig2 = plotly.io.from_json("/Users/toby/Dev/lionel-app/data/city_arsenal.json")
fig2

JSONDecodeError: Expecting value: line 1 column 1 (char 0)